# Día 5: Introducción a Raspberry Pi con Python

¡Bienvenido al mundo de la electrónica con Python! En este notebook aprenderemos:

1.  Cómo instalar el sistema operativo en una Raspberry Pi.
2.  Qué son los pines GPIO y cómo usarlos.
3.  Cuatro proyectos prácticos:
    *   Prender un LED.
    *   Usar un Sensor de Movimiento (PIR).
    *   Medir distancias con un Sensor de Ultrasonido.
    *   Controlar luces Neopixel.

## 1. Instalación del Sistema Operativo

Para que tu Raspberry Pi funcione, necesita un sistema operativo (OS). El oficial se llama **Raspberry Pi OS** (antes Raspbian).

### Paso a Paso:

1.  **Descargar el Imager**: Ve a la pc y descarga [Raspberry Pi Imager](https://www.raspberrypi.com/software/) para Windows.
2.  **Conectar la MicroSD**: Inserta tu tarjeta MicroSD en tu computadora.
3.  **Abrir Raspberry Pi Imager**:
    *   **Choose OS**: Selecciona "Raspberry Pi OS (32-bit)" (es el recomendado para empezar).
    *   **Choose Storage**: Selecciona tu tarjeta MicroSD.
4.  **Configuración Avanzada (¡Importante!)**:
    *   Antes de darle a "Write", presiona el ícono de engranaje (configuración) o `Ctrl+Shift+X`.
    *   Activa **SSH** (para controlarla remotamente).
    *   Configura tu **Wi-Fi** (SSID y contraseña).
    *   Establece un **usuario y contraseña** (ej. `pi` / `raspberry`).
5.  **Escribir**: Dale a "Write" y espera a que termine.
6.  **Arrancar**: Saca la tarjeta, ponla en la Raspberry Pi y conéctala a la corriente.

## 2. Entendiendo los GPIO (General Purpose Input/Output)

La fila de pines en la Raspberry Pi se llama **GPIO**. Nos permite controlar componentes electrónicos.

### Conceptos Clave:

*   **Voltaje**: Los pines trabajan a **3.3V**. ¡Cuidado! Si les metes 5V puedes quemar la placa.
*   **Numeración**: Hay dos formas de referirse a los pines:
    *   `BOARD`: El número físico del pin (1, 2, 3... hasta 40).
    *   `BCM`: El número del procesador Broadcom (GPIO2, GPIO3, etc.). **Usaremos BCM** porque es el estándar en la mayoría de tutoriales.
*   **Librería**: Usaremos `RPi.GPIO` que viene preinstalada en Raspberry Pi OS.

### Configuración Base en Python:

```python
import RPi.GPIO as GPIO
import time

GPIO.setmode(GPIO.BCM) # Usamos numeración BCM
GPIO.setwarnings(False)
```

## Proyecto 1: Prender un LED (Salida Digital)

Vamos a encender y apagar un LED.

### Materiales:
*   1 LED
*   1 Resistencia de 220 o 330 ohms (Obligatorio para no quemar el LED)
*   Cables macho-hembra

### Conexión:
*   Pata larga del LED (+) -> Pin GPIO 17
*   Pata corta del LED (-) -> Resistencia -> GND (Tierra física de la Raspberry)

### Código:

In [ ]:
import RPi.GPIO as GPIO
import time

# Configuración
LED_PIN = 17
GPIO.setmode(GPIO.BCM)
GPIO.setup(LED_PIN, GPIO.OUT) # Configuramos el pin como SALIDA

try:
    print("Parpadeando LED... (Presiona Ctrl+C para salir)")
    while True:
        GPIO.output(LED_PIN, True)  # Prender (3.3V)
        time.sleep(1)               # Esperar 1 segundo
        GPIO.output(LED_PIN, False) # Apagar (0V)
        time.sleep(1)
except KeyboardInterrupt:
    print("Saliendo...")
finally:
    GPIO.cleanup() # Limpia los pines al terminar

## Proyecto 2: Sensor de Movimiento PIR (Entrada Digital)

El sensor PIR detecta cambios en la radiación infrarroja (calor) para saber si algo se movió.

### Conexión:
*   VCC -> Pin 5V (El PIR necesita 5V para funcionar, aunque su señal de salida es de 3.3V, lo cual es seguro)
*   GND -> GND
*   OUT (Señal) -> GPIO 27

### Código:

In [ ]:
import RPi.GPIO as GPIO
import time

PIR_PIN = 27
GPIO.setmode(GPIO.BCM)
GPIO.setup(PIR_PIN, GPIO.IN) # Configuramos como ENTRADA

print("Esperando movimiento...")
try:
    while True:
        if GPIO.input(PIR_PIN):
            print("¡Movimiento Detectado! ")
            time.sleep(1) # Esperar un poco para no spamear
        else:
            print("Ningun movimiento detectado")
        time.sleep(0.1)
except KeyboardInterrupt:
    GPIO.cleanup()

## Proyecto 3: Sensor de Ultrasonido HC-SR04

Mide la distancia lanzando un sonido y contando cuánto tarda en rebotar.

**ADVERTENCIA**: El pin `ECHO` del sensor devuelve 5V. La Raspberry Pi soporta máximo 3.3V en sus pines de entrada. 
*   **Opción Segura**: Usar un divisor de voltaje (dos resistencias) en el pin ECHO.
*   **Opción Rápida (Riesgosa)**: Conectar una resistencia de 1k entre el ECHO del sensor y el GPIO de la Pi.

### Conexión:
*   VCC -> 5V
*   GND -> GND
*   TRIG -> GPIO 23
*   ECHO -> GPIO 24 (A través de resistencia/divisor de voltaje)

### Código:

In [ ]:
import RPi.GPIO as GPIO
import time

TRIG = 23
ECHO = 24

GPIO.setmode(GPIO.BCM)
GPIO.setup(TRIG, GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)

def medir_distancia():
    # 1. Asegurar que el Trigger esté bajo
    GPIO.output(TRIG, False)
    time.sleep(0.1)

    # 2. Enviar pulso de 10 microsegundos
    GPIO.output(TRIG, True)
    time.sleep(0.00001)
    GPIO.output(TRIG, False)

    # 3. Medir el tiempo del Echo
    inicio = time.time()
    fin = time.time()

    while GPIO.input(ECHO) == 0: # Esperar a que empiece el pulso
        inicio = time.time()
    
    while GPIO.input(ECHO) == 1: # Esperar a que termine el pulso
        fin = time.time()

    # 4. Calcular distancia
    # Tiempo * Velocidad del sonido (34300 cm/s) / 2 (ida y vuelta)
    duracion = fin - inicio
    distancia = (duracion * 34300) / 2
    return distancia

try:
    while True:
        dist = medir_distancia()
        print(f"Distancia: {dist:.1f} cm")
        time.sleep(1)
except KeyboardInterrupt:
    GPIO.cleanup()

## Proyecto 4: Tira LED Neopixel (WS2812B)

Los Neopixels son LEDs inteligentes donde puedes controlar el color de cada uno individualmente.

### Requisito previo (Instalar librería)
En la terminal de tu Raspberry Pi debes ejecutar esto para instalar la librería de Adafruit:
```bash
sudo pip3 install rpi_ws281x adafruit-circuitpython-neopixel
```
*(Nota: Este código debe ejecutarse como root/sudo en muchos casos)*

### Conexión:
*   5V -> 5V (Si son muchos LEDs, usa una fuente externa)
*   GND -> GND
*   DIN (Data In) -> GPIO 18 (PWM Pin)

### Código:

In [ ]:
import time
import board
import neopixel

# Configuración del Pixel
pixel_pin = board.D18
num_pixels = 8 # Número de LEDs en tu tira
ORDER = neopixel.GRB # Orden de colores (A veces es RGB)

pixels = neopixel.NeoPixel(
    pixel_pin, num_pixels, brightness=0.2, auto_write=False, pixel_order=ORDER
)

def arcoiris(wait):
    for j in range(255):
        for i in range(num_pixels):
            pixel_index = (i * 256 // num_pixels) + j
            pixels[i] = wheel(pixel_index & 255)
        pixels.show()
        time.sleep(wait)

def wheel(pos):
    # Función auxiliar para generar colores del arcoiris
    if pos < 0 or pos > 255:
        return (0, 0, 0)
    if pos < 85:
        return (255 - pos * 3, pos * 3, 0)
    if pos < 170:
        pos -= 85
        return (0, 255 - pos * 3, pos * 3)
    pos -= 170
        return (pos * 3, 0, 255 - pos * 3)

try:
    while True:
        print("Arcoiris...")
        arcoiris(0.01)
except KeyboardInterrupt:
    pixels.fill((0, 0, 0))
    pixels.show()